In [35]:
import pandas as pd
import warnings 
import json
import os
warnings.filterwarnings('ignore')

In [36]:
city = 'Bangalore'

df = pd.read_json(rf'../web scraping/{city}/car_dataset_{city.lower()}.json', lines=True)
df.head()

,url,car_name,Price,Registration Year,Insurance,Fuel Type,Seats,Kms Driven,RTO,Ownership,...,Charging Time,Fast Charging,Range - Tested,3rd Gear (30-80kmph),4th Gear (40-100kmph),Petrol Mileage WLTP,CNG Mileage ARAI,CNG Fuel Tank Capacity,Petrol Fuel Tank Capacity (Litres),Petrol Overall Mileage
0,https://www.cardekho.com/used-car-details/used...,Skoda Superb,₹26.65 Lakh,Jan 2021,Comprehensive,Petrol,5 Seats,"54,626 Kms",Bangalore,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.cardekho.com/buy-used-car-details/...,Kia Seltos,₹10.37 Lakh,Jan 2021,Comprehensive,Petrol,5 Seats,"50,756 Kms",Bangalore,Second Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.cardekho.com/buy-used-car-details/...,Maruti Suzuki Grand Vitara,₹12.82 Lakh,Dec 2022,Third Party,Petrol,5 Seats,"18,513 Kms",Bangalore,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.cardekho.com/buy-used-car-details/...,Maruti Suzuki Grand Vitara,₹11.04 Lakh,May 2023,Comprehensive,Petrol,5 Seats,"38,543 Kms",Bangalore,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.cardekho.com/used-car-details/used...,Tata Safari,₹23.50 Lakh,2023,-,Diesel,7 Seats,"30,000 Kms",Bangalore,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
PROCESSED_FILES_LOG = 'processed_files.txt'

def get_processed_files():
    if os.path.exists(PROCESSED_FILES_LOG):
        with open(PROCESSED_FILES_LOG, 'r') as f:
            return f.read().splitlines()
    return []

def mark_file_as_processed(json_filename):
    processed = get_processed_files()
    if json_filename not in processed:
        with open(PROCESSED_FILES_LOG, 'a') as f:
            f.write(json_filename + '\n')

In [38]:
# ─────────────────────────────────────────
json_file = f'car_dataset_{city.lower()}.json'

if json_file in get_processed_files():
    print(f"⚠️ '{json_file}' already processed — skipping!")
    df = pd.read_csv('dataset.csv')  
    print(f"Loaded existing dataset with {len(df)} rows")
    
else:  
    req_col = []
    with open('features.txt', 'r') as f:
        features = f.read().split('\n')
    for i in features:
        if i in df.columns:
            req_col.append(i) 
    df = df[req_col]

    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicates removed from new data: {before - len(df)} rows")

    if os.path.exists('dataset.csv') and os.path.getsize('dataset.csv') > 0:
        existing_df = pd.read_csv('dataset.csv')
        combined_df = pd.concat([existing_df, df], ignore_index=True)

        before = len(combined_df)
        combined_df = combined_df.drop_duplicates()
        print(f"Duplicates removed from combined data: {before - len(combined_df)} rows")

        combined_df.to_csv('dataset.csv', index=False)
        print(f"Appended {len(df)} rows. Total rows: {len(combined_df)}")
    else:
        df.to_csv('dataset.csv', index=False)
        print(f"Created new dataset.csv with {len(df)} rows")

    mark_file_as_processed(json_file)
    print(f"✅ '{json_file}' marked as processed!")

⚠️ 'car_dataset_bangalore.json' already processed — skipping!
Loaded existing dataset with 2261 rows


In [39]:
df = pd.read_csv('dataset.csv')

In [40]:
df.head()

,Mileage,Engine,Kerb Weight,Fuel,Transmission Type,Power,No. of Cylinders,Registration Year
0,18.9 kmpl,1197 cc,935 kg,Petrol,Manual,82 bhp,4.0,2015
1,19.81 kmpl,1086 cc,860 kg,Petrol,Manual,68.05 bhp,4.0,Apr 2015
2,15.6 kmpl,1196 cc,1090 kg,Petrol,Manual,70 bhp,4.0,Dec 2019
3,18.9 kmpl,1197 cc,1060 kg,Petrol,Manual,81.86 bhp,4.0,Jul 2017
4,25.44 kmpl,936 cc,1025 kg,Diesel,Manual,56.3 bhp,3.0,2015


In [41]:
df.shape

(2261, 8)

In [42]:
# Missing percentage count
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100
    print(f"{col}: {missing_percentage:.2f}% missing")

Mileage: 8.67% missing
Engine: 0.80% missing
Kerb Weight: 8.01% missing
Fuel: 12.69% missing
Transmission Type: 0.09% missing
Power: 2.48% missing
No. of Cylinders: 0.49% missing
Registration Year: 0.22% missing


In [43]:
df['Fuel'].value_counts()

Fuel
Petrol    1507
Diesel     436
CNG         31
Name: count, dtype: int64

1. Mileage

In [44]:
df['Mileage'] = df['Mileage'].str.extract(r'(\d+\.?\d*)').astype(float)

2. Engine

In [45]:
df['Engine'] = df['Engine'].str.extract(r'(\d+)').astype(float)

3. Weight

In [46]:
df['Kerb Weight'] = df['Kerb Weight'].str.extract(r'(\d+)').astype(float)

4. Power

In [47]:
df['Power'] = df['Power'].str.extract(r'(\d+\.?\d*)').astype(float)

5. Registration Year

In [48]:
df['Registration Year'] = pd.to_numeric(
    df['Registration Year'].str.extract(r'(\d{4})')[0],
    errors='coerce'
)

6. Transmission Type

In [49]:
df['Transmission Type'] = df['Transmission Type'].map({
                                'Automatic': 1,
                                'Manual': 0
                            })

7. Fuel

In [50]:
def encode_and_show_dropped(df, column, drop_first=True):
    dummies = pd.get_dummies(df[column], prefix=column, drop_first=drop_first)
    
    all_categories = df[column].unique()
    encoded_categories = [col.replace(f"{column}_", "") for col in dummies.columns]
    dropped_categories = [cat for cat in all_categories if cat not in encoded_categories]
    
    print(f"All categories:     {list(all_categories)}")
    print(f"Encoded categories: {encoded_categories}")
    print(f"Dropped categories: {dropped_categories}")
    
    return dummies

# usage
fuel_dummies = encode_and_show_dropped(df, 'Fuel', drop_first=True)
df = pd.concat([df.drop(columns=['Fuel']), fuel_dummies], axis=1)


All categories:     ['Petrol', 'Diesel', nan, 'CNG']
Encoded categories: ['Diesel', 'Petrol']
Dropped categories: [nan, 'CNG']


In [51]:
df['Mileage'].fillna(df['Mileage'].median(), inplace=True)

In [52]:
df.columns

Index(['Mileage', 'Engine', 'Kerb Weight', 'Transmission Type', 'Power',
       'No. of Cylinders', 'Registration Year', 'Fuel_Diesel', 'Fuel_Petrol'],
      dtype='object')

---

In [53]:
from sklearn.model_selection import train_test_split, GridSearchCV 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [54]:
# Features and target
X = df.drop(columns=['Mileage'])
y = df['Mileage']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [55]:
model = RandomForestRegressor()

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],        
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

# Grid search
grid_search = GridSearchCV(
    model,                            
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print("R2 Score:", r2)

Best parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best CV score: 0.8270078590501713
R2 Score: 0.8828856659387373


In [56]:
import os
import joblib
os.makedirs("model", exist_ok=True)
joblib.dump(best_model, "model/mileage_model.pkl")
print("Model saved!")


Model saved!


In [57]:
X_train.head(1)

,Engine,Kerb Weight,Transmission Type,Power,No. of Cylinders,Registration Year,Fuel_Diesel,Fuel_Petrol
610,1995.0,1500.0,1.0,184.0,4.0,2014.0,True,False
